In [4]:
import cv2
import numpy as np
import easyocr
from ultralytics import YOLO
from PIL import ImageFont, ImageDraw, Image

def run_license_plate_recognition():
    # 1. 모델 및 OCR 엔진 로드
    model_path = r'D:\미니프로젝트\plate_detection\01_detection\korea_plate_11small\weights\best.pt'
    model = YOLO(model_path)
    reader = easyocr.Reader(['ko', 'en'], gpu=True) 

    # 2. 영상 경로
    video_path = r'D:\미니프로젝트\test_video\KakaoTalk_20260309_145747748.mp4'
    cap = cv2.VideoCapture(video_path)

    # 3. 폰트 설정
    font_path = "C:/Windows/Fonts/malgun.ttf"
    font = ImageFont.truetype(font_path, 20)

    # [중요] 중복 출력 방지 변수를 while 루프 밖에서 초기화합니다.
    last_printed_text = ""

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        # 고해상도 추론
        results = model(frame, imgsz=640, conf=0.65)[0]

        for box in results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = box.conf[0]
            
            plate_roi = frame[max(0, y1-5):y2+5, max(0, x1-5):x2+5]
            if plate_roi.size == 0: continue

            plate_roi = cv2.resize(plate_roi, None, fx=2, fy=2, interpolation=cv2.INTER_LANCZOS4)
            plate_roi = cv2.detailEnhance(plate_roi, sigma_s=10, sigma_r=0.15)
            
            ocr_result = reader.readtext(plate_roi)
            # 글자만 추출 및 공백 제거
            plate_text = "".join([text.replace(" ", "") for (_, text, _) in ocr_result])

            # --- [수정된 출력 로직] ---
            if plate_text: # 글자가 인식되었을 때만 실행
                if plate_text != last_printed_text:
                    print(f"[인식 성공] 번호판: {plate_text} (신뢰도: {conf:.2f})")
                    last_printed_text = plate_text # 이제 에러 없이 갱신됩니다.
            # --------------------------

            # 화면 그리기 로직
            color = (0, 255, 0)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            
            img_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            draw = ImageDraw.Draw(img_pil)
            draw.text((x1, y1 - 30), f"{plate_text} ({conf:.2f})", font=font, fill=(0, 255, 0))
            frame = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)

        cv2.imshow('Final Korean Plate Recognition', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'): break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    run_license_plate_recognition()

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.



0: 640x384 4 cars, 93.1ms
Speed: 2.4ms preprocess, 93.1ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 4 cars, 110.7ms
Speed: 2.5ms preprocess, 110.7ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 4 cars, 96.0ms
Speed: 2.0ms preprocess, 96.0ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 4 cars, 97.9ms
Speed: 2.0ms preprocess, 97.9ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 4 cars, 89.4ms
Speed: 1.8ms preprocess, 89.4ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 4 cars, 93.9ms
Speed: 2.4ms preprocess, 93.9ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 4 cars, 105.4ms
Speed: 2.4ms preprocess, 105.4ms inference, 0.9ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 5 cars, 106.9ms
Speed: 2.5ms preprocess, 106.9ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 384)

0

In [5]:
import cv2
import numpy as np
import re
from ultralytics import YOLO
from paddleocr import PaddleOCR
from PIL import ImageFont, ImageDraw, Image

def run_lpr_advanced():
    # 1. YOLOv11s 모델 로드
    model = YOLO(r'D:\미니프로젝트\plate_detection\01_detection\korea_plate_11nano\weights\best.pt')
    
    # 2. PaddleOCR 로드 (에러 수정: 인자 최소화)
    # 'ValueError'를 방지하기 위해 문제가 되는 인자들을 제거했습니다.
    ocr_engine = PaddleOCR(lang='korean') 

    video_path = r'D:\미니프로젝트\test_video\26차_2861.jpg'
    cap = cv2.VideoCapture(video_path)

    # 한글 폰트 설정
    font = ImageFont.truetype("C:/Windows/Fonts/malgun.ttf", 20)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        # 고해상도 추론
        results = model(frame, imgsz=640, conf=0.5)[0]

        for box in results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = box.conf[0]
            
            # 번호판 영역 추출
            plate_roi = frame[max(0, y1-5):y2+5, max(0, x1-5):x2+5]
            if plate_roi.size == 0: continue
            
            # 확대 전처리
            plate_roi = cv2.resize(plate_roi, None, fx=2, fy=2, interpolation=cv2.INTER_LANCZOS4)

            # --- [PaddleOCR 인식] ---
            ocr_res = ocr_engine.ocr(plate_roi, cls=False)
            raw_text = ""
            if ocr_res and ocr_res[0]:
                for line in ocr_res[0]:
                    raw_text += line[1][0]

            # --- [번호판 형식 보정] ---
            clean_text = re.sub(r'[^0-9가-힣]', '', raw_text)
            match = re.search(r'(\d{2,3})([가-힣])(\d{4})', clean_text)
            final_text = f"{match.group(1)}{match.group(2)} {match.group(3)}" if match else clean_text

            # --- [화면 출력] ---
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            img_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            draw = ImageDraw.Draw(img_pil)
            draw.text((x1, y1 - 35), f"{final_text} ({conf:.2f})", font=font, fill=(0, 255, 0))
            frame = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)

        cv2.imshow('PaddleOCR LPR System', frame)
        if cv2.waitKey(0) & 0xFF == ord('q'): break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    run_lpr_advanced()

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\미니프로젝트\\plate_detection\\01_detection\\korea_plate_11nano\\weights\\best.pt'